In [2]:
# Imports
import os
from dotenv import load_dotenv

import numpy as np
from tqdm.notebook import tqdm

from huggingface_hub import login

import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


from pricer.items import Item
from pricer.evaluator import evaluate

from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.model_selection import train_test_split

In [3]:
LITE_MODE = True
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


In [4]:
LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.model(x)

In [7]:
y = np.array([float(item.price) for item in train])

documents = [item.summary for item in train]

In [12]:
np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [13]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.1, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [14]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 1,321,473


In [15]:
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 4

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/282 [00:00<?, ?it/s]

Epoch [1/4], Train Loss: 14520.564, Val Loss: 20701.260


  0%|          | 0/282 [00:00<?, ?it/s]

Epoch [2/4], Train Loss: 36136.926, Val Loss: 18430.258


  0%|          | 0/282 [00:00<?, ?it/s]

Epoch [3/4], Train Loss: 14843.091, Val Loss: 18156.240


  0%|          | 0/282 [00:00<?, ?it/s]

Epoch [4/4], Train Loss: 10365.348, Val Loss: 18454.918


In [16]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [17]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$59 $46 $19 $22 $45 $165 $5 $69 $19 $113 $344 $79 $74 $223 $8 $23 $13 $14 $93 $56 $16 $7 $96 $16 $275 $247 $302 $36 $49 $47 $53 $121 $31 $22 $166 $170 $58 $147 $123 $52 $167 $46 $7 $91 $129 $50 $70 $77 $12 $30 $6 $63 $48 $39 $119 $59 $35 $89 $19 $38 $53 $11 $18 $28 $440 $32 $2 $289 $25 $273 $5 $20 $148 $88 $20 $57 $20 $56 $28 $48 $69 $142 $8 $33 $13 $64 $53 $66 $131 $184 $16 $79 $17 $16 $31 $81 $37 $98 $142 $232 $11 $80 $5 $47 $23 $17 $96 $239 $3 $28 $23 $56 $102 $29 $1 $225 $188 $51 $69 $30 $14 $365 $16 $24 $79 $23 $17 $167 $54 $32 $24 $116 $99 $27 $61 $15 $107 $98 $53 $87 $34 $137 $4 $179 $207 $82 $64 $318 $78 $4 $13 $217 $10 $55 $20 $140 $133 $4 $1 $20 $73 $7 $5 $19 $513 $9 $4 $20 $8 $40 $14 $19 $204 $26 $13 $6 $15 $19 $23 $81 $366 $5 $73 $1 $12 $66 $28 $41 $27 $3 $50 $63 $66 $61 $7 $20 $78 $23 $12 $17 